# Demand shock with sticky wages: the planner's problem

The nominal-rate sequence is chosen to **maximise household utility** rather than to
minimise the output gap. Two scenarios:

1. **No time to build** — every firm reprices immediately.
2. **Exogenous time to build** — firms reprice with the calibrated lag $\tau$.

Companion to `new_gap.ipynb`, which minimises $\sum_t x_t^2$ instead.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import itertools

## 1. Calibration

In [ ]:
sigma = 5
beta = 1 / (1.0 + 0.01)
beta1 = 1 / (1.0 + 0.01 * np.exp(30/(36.5*0.9)))
beta_hh = beta
W = 1

gamma = 2.0
eta = 2.0
PHI_YA = (1 + eta) / (gamma + eta)
rho = 1 / beta_hh - 1

markup = sigma / (sigma - 1)

# Does a subsidy offset the monopolistic markup, making the steady state efficient?
# True  -> chi = 1      : steady state is first best; the planner only stabilises.
# False -> chi = 1/mu   : steady state is too small, so the planner ALSO wants to expand.
#                         That is a FIRST-ORDER effect and it swamps stabilisation.
SUBSIDY_OFFSETS_MARKUP = True
chi = 1.0 if SUBSIDY_OFFSETS_MARKUP else 1.0 / markup
chi_alt = 1.0 / markup if SUBSIDY_OFFSETS_MARKUP else 1.0

DEMAND_IN_UTILITY = False      # see the note below

print(f'gamma={gamma}, eta={eta}, phi_ya={PHI_YA:.4f}')
print(f'markup={markup:.4f}, chi={chi:.4f}  (N_ss = C_ss = A_ss = 1)')
print(f'efficient N at this chi: {chi**(-1/(gamma+eta)):.4f}   '
      f'(1.0000 => steady state is first best)')

In [ ]:
data = pd.read_csv('APP_data_usa_goods.csv', index_col=0, parse_dates=True)
data_tau = (data['invtCogsRatio'] * 36.5 / (30 * 0.1))
plt.hist(data_tau, bins=55)
plt.title('Distribution of Inventory Adjustment Times')
plt.xlabel('Adjustment Time (tau)'); plt.ylabel('Frequency')
plt.show()

In [ ]:
tau_diff = np.zeros((1000, 1))
for i, c in enumerate(np.linspace(0, 0.99, 1000)):
    if beta * c <= 1e-10:
        tau_diff[i] = np.inf
        continue
    N = -(data_tau) - 1/np.log(beta * c)
    tau1 = -1/np.log(beta * c) - N;   tau1[tau1 < 0] = 0
    tau2 = -1/np.log(beta1 * c) - N;  tau2[tau2 < 0] = 0
    tau_diff[i] = np.mean(np.abs(tau1 - tau2))

c_list = np.linspace(0, 0.99, 1000)
c = np.min(c_list[np.where(np.abs(tau_diff - 1) == np.min(np.abs(tau_diff - 1)))[0]])
print('optimal c:', c)

In [ ]:
N = -data_tau - 1/np.log(beta * c)
tau = -1/np.log(beta * c) - N
tau[tau < 0] = 0
tau_floor, tau_ceil = np.floor(tau), np.ceil(tau)
tprod_ceil = (beta * c)**tau_ceil * (tau_ceil + N)
tprod_floor = (beta * c)**tau_floor * (tau_floor + N)
tau[(tprod_ceil - tprod_floor) > 0] = tau_ceil[(tprod_ceil - tprod_floor) > 0]
tau[(tprod_ceil - tprod_floor) < 0] = tau_floor[(tprod_ceil - tprod_floor) < 0]
tau[tau < 0] = 0

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(N, bins=50); ax[0].set_title('Distribution of N'); ax[0].set_xlabel('N')
ax[1].hist(tau, bins=55); ax[1].set_title('Distribution of Time to Produce'); ax[1].set_xlabel('tau')
plt.tight_layout(); plt.show()

## 2. The IS curve

$$x_t = x_{t+1} - \frac{1}{\gamma}\bigl(i_t - \pi_{t+1} - r^n_t\bigr), \qquad
  r^n_t = \rho - d_t + \gamma\,(y^n_{t+1}-y^n_t)$$

In [ ]:
def euler_path_gap_2(i_path, price_level, demand_level, a_nat, gamma=2.0, eta=2.0,
                     beta_hh=1/(1+0.01)):
    """IS curve in output-gap form with a demand shock.  Returns (x, pi, y_nat, r_nat)."""
    i_path = np.asarray(i_path, float)
    price_level = np.asarray(price_level, float)
    demand_level = np.asarray(demand_level, float)
    a_nat = np.asarray(a_nat, float)

    T = len(i_path)
    rho = 1.0 / beta_hh - 1.0
    phi_ya = (1.0 + eta) / (gamma + eta)

    pi = np.zeros(T)
    pi[1:] = price_level[1:] - price_level[:-1]
    pi_fwd = np.r_[pi[1:], 0.0]

    y_nat = phi_ya * a_nat
    dy_nat_fwd = np.r_[y_nat[1:] - y_nat[:-1], 0.0]
    r_nat = rho - demand_level + gamma * dy_nat_fwd

    x = np.zeros(T + 1)
    for t in range(T - 1, -1, -1):
        x[t] = x[t + 1] - (1.0 / gamma) * (i_path[t] - pi_fwd[t] - r_nat[t])

    return x[:T], pi, y_nat, r_nat

## 3. The planner's objective

The planner picks the nominal-rate sequence that **maximises household utility**

$$U = \sum_{t=0}^{T-1}\beta^t\Bigl[\frac{C_t^{1-\gamma}}{1-\gamma}
      - \chi\frac{N_t^{1+\eta}}{1+\eta}\Bigr]$$

evaluated on the model's own paths, rather than minimising an ad-hoc quadratic loss.
Given the log deviations $y_t$ and $a_t$:

$$C_t = e^{y_t} \quad (C=Y), \qquad N_t = e^{y_t-a_t} \quad (y = a + n)$$

$\chi$ is pinned down by the steady-state labour FOC. At the monopolistically competitive
steady state the real wage is $1/\mathcal{M}$ with $\mathcal{M}=\sigma/(\sigma-1)$, and
normalising $A=C=N=1$ gives $\chi = 1/\mathcal{M} = (\sigma-1)/\sigma$. Note this steady
state is **not** the first best: the markup makes $N_{ss}$ too low, so the planner here is
a constrained one — it chooses rates taking the markup and the wage rigidity as given.

Welfare differences are reported as a **consumption equivalent** $\lambda$: the permanent
percentage of steady-state consumption that would have to be added to the no-policy-change
path to make it as good as the alternative.

Each experiment also reports the gap-minimising sequence from the companion notebook, so
the two objectives can be compared directly.

### How the demand shock enters utility

The demand shock $d_t$ enters the IS curve additively; the notebook it came from never
specifies a microfoundation, and that matters for a welfare criterion. Two readings:

* `DEMAND_IN_UTILITY = False` — $d_t$ is a pure disturbance to the IS curve and does not
  appear in the period utility function. Utility weights are the usual $\beta^t$.
* `DEMAND_IN_UTILITY = True` — $d_t$ is a genuine preference shock. Matching the IS curve
  $c_t = c_{t+1} - \frac1\gamma(i_t-\pi_{t+1}-\rho+d_t)$ to the Euler equation of
  $\sum\beta^t\xi_t\,u(C_t,N_t)$ requires $d_t = \Delta\log\xi_{t+1}$, so a one-period
  $d_0 = 0.01$ implies a *permanent* level shift $\log\xi_t = 0.01$ for all $t\ge 1$.

Both are computed below and reported side by side. They shift the level of utility but,
as the output shows, need not change which sequence is chosen.

In [ ]:
def welfare(y_path, a_path, gamma, eta, beta_hh, chi, xi_log=None):
    """
    Discounted household utility along a model path.

        U = sum_t beta^t * xi_t * [ C_t^(1-gamma)/(1-gamma) - chi * N_t^(1+eta)/(1+eta) ]

    y_path, a_path are LOG DEVIATIONS from a steady state normalised to A=C=N=1, so

        C_t = exp(y_t)                      goods market clearing, C = Y
        N_t = exp(y_t - a_t)                production function y = a + n

    xi_log is the log preference shifter (None -> xi = 1 throughout).
    Returns (U, U_consumption_part, U_labour_part).
    """
    y_path = np.asarray(y_path, float)
    a_path = np.asarray(a_path, float)
    T = len(y_path)
    disc = beta_hh ** np.arange(T)
    if xi_log is not None:
        disc = disc * np.exp(np.asarray(xi_log, float))

    C = np.exp(y_path)
    Nl = np.exp(y_path - a_path)

    u_c = C**(1 - gamma) / (1 - gamma)
    u_n = chi * Nl**(1 + eta) / (1 + eta)
    return np.sum(disc * (u_c - u_n)), np.sum(disc * u_c), np.sum(disc * u_n)


def cons_equivalent(U_alt, UC_base, UN_base, gamma):
    """
    Permanent consumption supplement lambda that makes the BASELINE path as good as
    the alternative:  sum beta^t [((1+l)C_b)^(1-g)/(1-g)] - UN_base = U_alt.
    Returns lambda (0.01 = 1 percent of steady-state consumption).
    """
    ratio = (U_alt + UN_base) / UC_base
    return ratio**(1.0 / (1.0 - gamma)) - 1.0


def summarise_planner(label, U, r_arrange, matches, r_choice, X_gap, dev_gap,
                      UC, UN, gamma, U_alt=None, alt_label=''):
    """Standard summary: utility-optimal vs gap-optimal vs no policy change."""
    U = np.asarray(U).ravel()
    dev_gap = np.asarray(dev_gap).ravel()
    best_u = int(np.argmax(U))                 # planner: MAXIMISE utility
    best_x = int(np.argmin(dev_gap))           # gap-minimising benchmark
    base = int(np.where(matches)[0][0])
    w = 46
    lam_u = cons_equivalent(U[best_u], UC[base], UN[base], gamma)
    lam_x = cons_equivalent(U[best_x], UC[base], UN[base], gamma)
    print('=' * 74)
    print(label)
    print('=' * 74)
    print(f'{"utility, no policy change":<{w}}: {U[base]:.10f}')
    print(f'{"utility, utility-maximising policy":<{w}}: {U[best_u]:.10f}')
    print(f'{"utility, gap-minimising policy":<{w}}: {U[best_x]:.10f}')
    print()
    print(f'{"welfare gain, utility-max (cons. equiv.)":<{w}}: {100*lam_u:+.6f}% of ss C')
    print(f'{"welfare gain, gap-min   (cons. equiv.)":<{w}}: {100*lam_x:+.6f}% of ss C')
    print()
    print(f'{"rate sequence, no policy change":<{w}}: {np.round(r_arrange[base], 6)}')
    print(f'{"rate sequence, utility-maximising":<{w}}: {np.round(r_arrange[best_u], 6)}')
    print(f'{"rate sequence, gap-minimising":<{w}}: {np.round(r_arrange[best_x], 6)}')
    print(f'{"utility-max and gap-min coincide":<{w}}: {best_u == best_x}')
    if U_alt is not None:
        alt = int(np.argmax(np.asarray(U_alt).ravel()))
        print(f'{"rate sequence, " + alt_label:<{w}}: {np.round(r_arrange[alt], 6)}')
        print(f'{"  ... same choice as the baseline":<{w}}: {alt == best_u}')
    print()
    print(f'{"peak |x_t|, no policy change":<{w}}: {np.max(np.abs(X_gap[base])):.4e}')
    print(f'{"peak |x_t|, utility-maximising":<{w}}: {np.max(np.abs(X_gap[best_u])):.4e}')
    on_bound = bool(np.any(np.isclose(r_arrange[best_u], r_choice[0]) |
                           np.isclose(r_arrange[best_u], r_choice[-1])))
    print(f'{"grid":<{w}}: {len(r_choice)} points on '
          f'[{r_choice[0]:g}, {r_choice[-1]:g}]')
    print(f'{"optimum sits on a grid bound":<{w}}: {on_bound}')
    if on_bound:
        print('   -> the optimum may not be interior; widen r_choice to check.')
    return best_u, best_x, base


def planner_figure(label, X_gap, I_path, best_u, best_x, base, fname, horizon=20):
    """Two panels on a common horizon: the output gap and the nominal rate."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    t = np.arange(horizon)

    axes[0].plot(t, X_gap[base][:horizon], lw=2, label='no policy change')
    axes[0].plot(t, X_gap[best_u][:horizon], '--', lw=2, label='utility-maximising policy')
    if best_x != best_u:
        axes[0].plot(t, X_gap[best_x][:horizon], ':', lw=2, label='gap-minimising policy')
    axes[0].axhline(0, color='k', lw=0.8)
    axes[0].set_xlabel('Time'); axes[0].set_ylabel('$x_t = y_t - y^n_t$')
    axes[0].set_xlim(0, horizon - 1)
    axes[0].set_title('Output gap')
    axes[0].legend(fontsize=8)

    axes[1].step(t, I_path[base][:horizon], where='post', lw=2, label='no policy change')
    axes[1].step(t, I_path[best_u][:horizon], where='post', linestyle='--', lw=2,
                 label='utility-maximising policy')
    if best_x != best_u:
        axes[1].step(t, I_path[best_x][:horizon], where='post', linestyle=':', lw=2,
                     label='gap-minimising policy')
    axes[1].set_xlabel('Time'); axes[1].set_ylabel('$i_t$')
    axes[1].set_xlim(0, horizon - 1)
    axes[1].set_title('Nominal interest rate')
    axes[1].legend(fontsize=8)

    fig.suptitle(label, fontsize=13)
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def xi_from_demand(demand, use_it):
    """log preference shifter implied by d_t = Delta log xi_{t+1}, or None."""
    if not use_it:
        return None
    T = len(demand)
    xi_log = np.zeros(T)
    for t in range(1, T):
        xi_log[t] = xi_log[t-1] + demand[t-1]
    return xi_log

## 4. Planner's problem: no time to build

No TFP movement, so potential output is flat and the whole shock lands in the natural rate.

In [ ]:
T = 10
sequence_length = 3
r_choice = np.linspace(0, 0.02, 9)
r_arrange = np.array(list(itertools.product(r_choice, repeat=sequence_length)))
size = r_arrange.shape[0]

U_total = np.zeros(size); U_cons = np.zeros(size); U_lab = np.zeros(size)
U_alt = np.zeros(size)                       # the other DEMAND_IN_UTILITY convention
U_chi_alt = np.zeros(size)                   # the other steady-state calibration
deviation_gap = np.zeros(size)
X_gap = np.zeros((size, T)); I_path_store = np.zeros((size, T))

for k in range(size):
    i_path = np.ones(T) * rho
    i_path[:sequence_length] = r_arrange[k]

    demand = np.zeros(T)
    demand[0] = 0.01                        # 1 percent demand shock
    a_path = np.zeros(T)

    x, pi, y_nat, r_nat = euler_path_gap_2(i_path, np.zeros(T), demand, a_path,
                                           gamma=gamma, eta=eta, beta_hh=beta_hh)
    y = x + y_nat

    U, Uc, Un = welfare(y, a_path, gamma, eta, beta_hh, chi,
                        xi_from_demand(demand, DEMAND_IN_UTILITY))
    Ua, _, _ = welfare(y, a_path, gamma, eta, beta_hh, chi,
                       xi_from_demand(demand, not DEMAND_IN_UTILITY))
    Uca, _, _ = welfare(y, a_path, gamma, eta, beta_hh, chi_alt,
                        xi_from_demand(demand, DEMAND_IN_UTILITY))

    U_total[k], U_cons[k], U_lab[k] = U, Uc, Un
    U_alt[k] = Ua
    U_chi_alt[k] = Uca
    deviation_gap[k] = np.sum(x**2)
    X_gap[k] = x
    I_path_store[k] = i_path

matches = np.all(np.isclose(r_arrange, rho), axis=1)
bu_ntb, bx_ntb, base_ntb = summarise_planner(
    '1 percent demand shock -- No time to build',
    U_total, r_arrange, matches, r_choice, X_gap, deviation_gap, U_cons, U_lab, gamma,
    U_alt=U_chi_alt, alt_label='under the other steady state')
print(f'{"same choice under the other xi convention":<46}: '
      f'{int(np.argmax(U_alt)) == bu_ntb}')

In [ ]:
planner_figure('1 percent demand shock: No time to build (planner)',
               X_gap, I_path_store, bu_ntb, bx_ntb, base_ntb,
               '../semi_endo_demand_planner_no_time_to_build.png', horizon=10)

## 5. Planner's problem: exogenous time to build

Potential output is rate-dependent: the rate path feeds firms' discounted production
`tprod`, so policy shifts $y^n$ through the time-to-produce channel.

In [ ]:
T = 30
sequence_length = 3
r_choice = np.linspace(0, 0.0125, 6)
r_arrange = np.array(list(itertools.product(r_choice, repeat=sequence_length)))
size = r_arrange.shape[0]

U_total = np.zeros(size); U_cons = np.zeros(size); U_lab = np.zeros(size)
U_alt = np.zeros(size)
U_chi_alt = np.zeros(size)                   # the other steady-state calibration
deviation_gap = np.zeros(size)
X_gap = np.zeros((size, T)); I_path_store = np.zeros((size, T))

tvals = tau.astype(int).to_numpy()
N_array = np.array(N)

for k in range(size):
    n = sequence_length
    r = np.asarray(r_arrange[k], dtype=float)

    interest_rate_list = np.ones((T, T)) * rho
    arr2d = np.full((n, n), rho, dtype=float)
    for i in range(n):
        arr2d[i, :i+1] = r[i::-1]
    interest_rate_list[0:n, :n] = arr2d

    beta_mat = 1.0 / (1.0 + interest_rate_list)
    cumprod_ext = np.ones((T, T + 1))
    cumprod_ext[:, 1:] = np.cumprod(beta_mat, axis=1)
    beta_prod = cumprod_ext[:, tvals]

    tprod_array = np.array((beta_prod * 0.9018018) *
                           ((tvals + N_array) * np.ones((T, 269))))

    A_price = np.sum(1/tprod_array**(sigma - 1), axis=1)**(1/(sigma - 1))
    A_tfp = np.sum(tprod_array**(sigma - 1), axis=1)**(1/(sigma - 1))

    i_path = np.ones(T) * rho
    i_path[0:sequence_length] = r_arrange[k]

    demand = np.zeros(T)
    demand[0] = 0.01
    a_nat = np.log(A_tfp) - np.log(A_tfp)[-1]

    x, pi, y_nat, r_nat = euler_path_gap_2(i_path, np.log(A_price), demand, a_nat,
                                           gamma=gamma, eta=eta, beta_hh=beta_hh)
    y = x + y_nat

    U, Uc, Un = welfare(y, a_nat, gamma, eta, beta_hh, chi,
                        xi_from_demand(demand, DEMAND_IN_UTILITY))
    Ua, _, _ = welfare(y, a_nat, gamma, eta, beta_hh, chi,
                       xi_from_demand(demand, not DEMAND_IN_UTILITY))
    Uca, _, _ = welfare(y, a_nat, gamma, eta, beta_hh, chi_alt,
                        xi_from_demand(demand, DEMAND_IN_UTILITY))

    U_total[k], U_cons[k], U_lab[k] = U, Uc, Un
    U_alt[k] = Ua
    U_chi_alt[k] = Uca
    deviation_gap[k] = np.sum(x**2)
    X_gap[k] = x
    I_path_store[k] = i_path

matches = np.all(np.isclose(r_arrange, rho), axis=1)
bu_ttb, bx_ttb, base_ttb = summarise_planner(
    '1 percent demand shock -- Exogenous time to build',
    U_total, r_arrange, matches, r_choice, X_gap, deviation_gap, U_cons, U_lab, gamma,
    U_alt=U_chi_alt, alt_label='under the other steady state')
print(f'{"same choice under the other xi convention":<46}: '
      f'{int(np.argmax(U_alt)) == bu_ttb}')

### Why the planner goes to the corner here

Unlike the no-time-to-build case, the utility optimum below is the **lowest rate on the
grid**, and it does *not* coincide with the gap optimum. That is the model, not a bug.

Firm output is $\text{tprod}_i = \beta^{\tau_i} c\,(\tau_i + N_i)$ with
$\beta = 1/(1+i)$: the nominal rate is a *cost of production* applied over the lag
$\tau_i$. So a permanent rate cut raises measured productivity by roughly
$\bar\tau\times(\text{the cut})$ — a **first-order level gain** with nothing in the
model to offset it. Exact utility picks that up; the quadratic gap objective, which only
sees deviations from potential, cannot.

Read the corner as a diagnostic: within this model monetary policy has a free, permanent
productivity lever, so "maximise utility" degenerates to "cut to the bound". If that is
not intended, the rate needs a cost somewhere (a resource cost of the lag, or firms
discounting at a rate policy does not control).

In [ ]:
# --- size of the rate -> productivity channel --------------------------------
CVAL = c
probe = {}
for i_lvl in [0.0, rho]:
    irl = np.ones((T, T)) * i_lvl
    ce = np.ones((T, T + 1)); ce[:, 1:] = np.cumprod(1/(1 + irl), axis=1)
    bp = ce[:, tvals]
    tp = (bp * CVAL) * ((tvals + N_array) * np.ones((T, 269)))
    probe[i_lvl] = np.sum(tp**(sigma - 1), axis=1)**(1/(sigma - 1))

gain = float(np.log(probe[0.0][0] / probe[rho][0]))
print(f'mean tau                                  : {float(tau.mean()):.3f} periods')
print(f'cutting i from rho={rho:.4f} to 0 raises A by : {100*gain:.4f}%')
print(f'compare  mean(tau) * rho                  : {100*float(tau.mean())*rho:.4f}%')

In [ ]:
planner_figure('1 percent demand shock: Exogenous time to build (planner)',
               X_gap, I_path_store, bu_ttb, bx_ttb, base_ttb,
               '../semi_endo_demand_planner_exogenous_time_to_build.png', horizon=10)

## Escalating the policy horizon

Every optimum above sits on a bound of `r_choice`, which means the answer is being shaped
by the grid and by the short window of policy freedom rather than by the economics. This
section grows the number of free periods $L$ — everything after $L$ stays pinned at $\rho$
— and re-solves, stopping when the **last free period comes back to $\rho$**. At that
point the horizon is long enough that pinning the tail no longer binds.

Grid enumeration is $6^L$ and blows up immediately, so this uses a bounded continuous
solve (L-BFGS-B on $[0,\,0.05]$, the lower bound being the ZLB) instead of enumerating.

In [ ]:
from scipy.optimize import minimize


def escalate(objective, T, L_start=3, L_max=16, tol=1e-4, bounds=(0.0, 0.05)):
    """
    Grow the number of free policy periods L until the optimal sequence's LAST free
    period returns to rho.  `objective(r_vec, T)` is MINIMISED (pass -utility for the
    planner problems).  Returns (history, settled_L); settled_L is None if it never
    settles by L_max.
    """
    hist = []
    print(f'{"L":>3} {"last i_t":>10} {"settled":>8} {"objective":>15}   optimal sequence')
    print('-' * 92)
    for L in range(L_start, L_max + 1):
        res = minimize(lambda r: objective(r, T), np.ones(L) * rho,
                       method='L-BFGS-B', bounds=[bounds] * L,
                       options={'maxiter': 20000, 'ftol': 1e-18, 'gtol': 1e-14})
        settled = bool(np.isclose(res.x[-1], rho, atol=tol))
        hist.append((L, res.x.copy(), float(res.fun), settled))
        print(f'{L:>3} {res.x[-1]:>10.6f} {str(settled):>8} {res.fun:>15.6e}   '
              f'{np.round(res.x[:8], 5)}{" ..." if L > 8 else ""}')
        if settled:
            print(f'\n-> settles at L = {L}: the last free period is back at '
                  f'rho = {rho:.4f}')
            return hist, L
    print(f'\n-> NOT settled by L = {L_max}: the optimum is still away from rho.')
    return hist, None


def escalate_figure(hist, settled_L, label, fname):
    """Objective against L, and the rate path at the final L."""
    Ls = [h[0] for h in hist]
    objs = [h[2] for h in hist]
    r_last = hist[-1][1]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    axes[0].plot(Ls, objs, 'o-')
    axes[0].set_yscale('symlog', linthresh=1e-18)
    axes[0].set_xlabel('free policy periods $L$'); axes[0].set_ylabel('objective')
    axes[0].set_title('Objective as the horizon grows')

    p = np.arange(len(r_last))
    axes[1].step(p, r_last, where='post', lw=2, label=f'optimal $i_t$ at L={Ls[-1]}')
    axes[1].axhline(rho, color='k', lw=0.8, linestyle='--', label=r'$\rho$')
    axes[1].set_xlabel('Period'); axes[1].set_ylabel('$i_t$')
    axes[1].set_title('Optimal rate path at the final $L$')
    axes[1].legend(fontsize=8)

    status = f'settles at L={settled_L}' if settled_L else 'does not settle'
    fig.suptitle(f'{label}  ({status})', fontsize=13)
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def price_tfp_from_rates(r, T):
    L = len(r)
    interest_rate_list = np.ones((T, T)) * rho
    arr2d = np.full((L, L), rho, dtype=float)
    for i in range(L):
        arr2d[i, :i+1] = np.asarray(r, float)[i::-1]
    interest_rate_list[0:L, :L] = arr2d
    ce = np.ones((T, T + 1))
    ce[:, 1:] = np.cumprod(1.0/(1.0 + interest_rate_list), axis=1)
    bp = ce[:, tvals]
    tp = (bp * 0.9018018) * ((tvals + N_array) * np.ones((T, 269)))
    A_price = np.sum(1/tp**(sigma - 1), axis=1)**(1/(sigma - 1))
    A_tfp = np.sum(tp**(sigma - 1), axis=1)**(1/(sigma - 1))
    return np.log(A_price), np.log(A_tfp) - np.log(A_tfp)[-1]


def obj_u_ntb(r, T):
    ip = np.ones(T) * rho; ip[:len(r)] = r
    d = np.zeros(T); d[0] = 0.01
    a = np.zeros(T)
    x, pi, y_nat, r_nat = euler_path_gap_2(ip, np.zeros(T), d, a,
                                           gamma=gamma, eta=eta, beta_hh=beta_hh)
    return -welfare(x + y_nat, a, gamma, eta, beta_hh, chi,
                    xi_from_demand(d, DEMAND_IN_UTILITY))[0]


def obj_u_ttb(r, T):
    ip = np.ones(T) * rho; ip[:len(r)] = r
    d = np.zeros(T); d[0] = 0.01
    pl, an = price_tfp_from_rates(r, T)
    x, pi, y_nat, r_nat = euler_path_gap_2(ip, pl, d, an,
                                           gamma=gamma, eta=eta, beta_hh=beta_hh)
    return -welfare(x + y_nat, an, gamma, eta, beta_hh, chi,
                    xi_from_demand(d, DEMAND_IN_UTILITY))[0]


print('NO TIME TO BUILD')
hist_ntb, L_ntb = escalate(obj_u_ntb, 10, L_start=3, L_max=9)
print()
print('EXOGENOUS TIME TO BUILD')
hist_ttb, L_ttb = escalate(obj_u_ttb, 30, L_start=3, L_max=16)

In [ ]:
escalate_figure(hist_ntb, L_ntb, '1% demand shock: no time to build (planner)',
                '../semi_endo_demand_planner_escalate_no_time_to_build.png')
escalate_figure(hist_ttb, L_ttb, '1% demand shock: exogenous time to build (planner)',
                '../semi_endo_demand_planner_escalate_exogenous_time_to_build.png')